In [ ]:
# Equal-Weight Baseline and Downside Risk

This notebook establishes the equal-weight portfolio as the baseline strategy.

The portfolio is rebalanced every 21 trading days and incorporates transaction costs based on portfolio turnover.

Performance is evaluated using:

- CAGR
- Annualized volatility
- Sharpe ratio
- Sortino ratio
- Maximum drawdown
- Historical 95% VaR
- Historical 95% CVaR
- Portfolio turnover

The Sortino ratio uses downside deviation relative to a zero target return.

In [10]:
import pandas as pd
import numpy as np
from pathlib import Path

# =====================================
# 1. Load data
# =====================================

data_dir = Path("../data")
output_dir = Path("../outputs")
output_dir.mkdir(exist_ok=True)

prices = pd.read_csv(
    data_dir / "adjusted_close.csv",
    index_col=0,
    parse_dates=True
)

assets = prices.columns.tolist()
N = len(assets)

print("Assets:", assets)
print("Number of assets:", N)


# =====================================
# 2. Portfolio backtest function
# =====================================

def backtest_fixed_weights(
    prices,
    target_weights,
    rebalance_every=21,
    transaction_cost=0.001,
    initial_value=1.0
):
    """
    Backtest a target-weight portfolio.

    Rebalances every `rebalance_every` trading days.
    Between rebalances, asset weights are allowed to drift.

    transaction_cost:
        Cost per unit of one-way portfolio turnover.
        0.001 = 10 basis points.
    """

    target_weights = pd.Series(
        target_weights,
        index=prices.columns,
        dtype=float
    )

    target_weights = target_weights / target_weights.sum()

    shares = None
    portfolio_values = []
    turnovers = []

    for i, (date, px) in enumerate(prices.iterrows()):

        px = px.astype(float)

        # Initial investment
        if shares is None:

            portfolio_value = initial_value

            shares = (
                portfolio_value
                * target_weights
                / px
            )

            turnover = 0.0

        else:

            portfolio_value = float(
                (shares * px).sum()
            )

            # Rebalance
            if i % rebalance_every == 0:

                current_values = shares * px

                current_weights = (
                    current_values
                    / portfolio_value
                )

                # One-way turnover
                turnover = (
                    0.5
                    * np.abs(
                        target_weights
                        - current_weights
                    ).sum()
                )

                trading_cost = (
                    portfolio_value
                    * transaction_cost
                    * turnover
                )

                portfolio_value -= trading_cost

                shares = (
                    portfolio_value
                    * target_weights
                    / px
                )

            else:

                turnover = 0.0

        portfolio_values.append(
            portfolio_value
        )

        turnovers.append(
            turnover
        )

    result = pd.DataFrame(
        {
            "Wealth": portfolio_values,
            "Turnover": turnovers
        },
        index=prices.index
    )

    result["Return"] = (
        result["Wealth"]
        .pct_change()
    )

    return result


# =====================================
# 3. Equal-weight portfolio
# =====================================

equal_weights = np.repeat(
    1 / N,
    N
)

ew_net = backtest_fixed_weights(
    prices,
    equal_weights,
    rebalance_every=21,
    transaction_cost=0.001
)

# Gross version for comparison
ew_gross = backtest_fixed_weights(
    prices,
    equal_weights,
    rebalance_every=21,
    transaction_cost=0.0
)


# =====================================
# 4. Risk/performance metrics
# =====================================

def portfolio_metrics(backtest):

    wealth = backtest["Wealth"]
    returns = backtest["Return"].dropna()

    years = (
        wealth.index[-1]
        - wealth.index[0]
    ).days / 365.25

    cagr = (
        wealth.iloc[-1]
        / wealth.iloc[0]
    ) ** (1 / years) - 1

    annual_vol = (
        returns.std()
        * np.sqrt(252)
    )

    # rf = 0 for this preliminary diagnostic.
    # Final risk-adjusted metrics use BIL in notebook 06.
    sharpe_rf0 = (
        returns.mean()
        / returns.std()
        * np.sqrt(252)
    )

    # Downside deviation relative to a zero target return
    downside_returns = np.minimum(
        returns,
        0
    )

    downside_deviation = (
        np.sqrt(
            np.mean(
                downside_returns ** 2
            )
        )
        * np.sqrt(252)
    )

    sortino_rf0 = (
        returns.mean() * 252
        / downside_deviation
    )

    running_max = wealth.cummax()

    drawdown = (
        wealth
        / running_max
        - 1
    )

    max_drawdown = (
        drawdown.min()
    )

    # Historical VaR / CVaR
    q05 = returns.quantile(0.05)

    var_95 = -q05

    cvar_95 = -returns[
        returns <= q05
    ].mean()

    annualized_turnover = (
        backtest["Turnover"].mean()
        * 252
    )

    return pd.Series(
        {
            "CAGR": cagr,
            "Annualized_Volatility": annual_vol,
            "Sharpe_rf0": sharpe_rf0,
            "Sortino_rf0": sortino_rf0,
            "Maximum_Drawdown": max_drawdown,
            "Daily_VaR_95": var_95,
            "Daily_CVaR_95": cvar_95,
            "Annualized_Turnover": annualized_turnover
        }
    )


metrics = pd.DataFrame(
    {
        "EqualWeight_Gross":
            portfolio_metrics(ew_gross),

        "EqualWeight_Net":
            portfolio_metrics(ew_net)
    }
).T

print(
    "\n=== Equal-Weight Benchmark ==="
)

display(
    metrics.round(4)
)


# =====================================
# 5. Wealth + drawdown
# =====================================

comparison = pd.DataFrame(
    {
        "Gross": ew_gross["Wealth"],
        "Net": ew_net["Wealth"]
    }
)

comparison[
    "Net_Drawdown"
] = (
    comparison["Net"]
    / comparison["Net"].cummax()
    - 1
)


# =====================================
# 6. Save
# =====================================

metrics.to_csv(
    output_dir
    / "equal_weight_metrics.csv"
)

comparison.to_csv(
    output_dir
    / "equal_weight_wealth.csv"
)

print(
    "\nFinal net wealth:",
    round(
        ew_net["Wealth"].iloc[-1],
        4
    )
)

print(
    "Number of rebalances:",
    int(
        (
            ew_net["Turnover"] > 0
        ).sum()
    )
)

Assets: ['SPY', 'QQQ', 'IWM', 'EFA', 'EEM', 'TLT', 'GLD', 'VNQ']
Number of assets: 8

=== Equal-Weight Benchmark ===


,CAGR,Annualized_Volatility,Sharpe_rf0,Sortino_rf0,Maximum_Drawdown,Daily_VaR_95,Daily_CVaR_95,Annualized_Turnover
EqualWeight_Gross,0.0976,0.134,0.7633,1.0740,-0.2718,0.0125,0.0197,0.1549
EqualWeight_Net,0.0974,0.134,0.7621,1.0723,-0.2719,0.0125,0.0197,0.1549



Final net wealth: 4.4223
Number of rebalances: 191
